# Camera calibration

The starting point, from `camera_model.ipynb`: the citizen model (Model 1), shifted 7° of magnetic latitude south,
matches the aurora rates the cameras actually recorded, more closely than any other shift. From geometry,
aurora overhead a camera 7° (~800 km) north still stands 4-5° above a person's northern horizon. So for each of the 9 cameras we take Model 1's clear-sky predicted probability 7° south of the camera and fit a map
`sigmoid(a * logit(p) + b)` to what that camera recorded. One (a, b) per camera. The maps are monotone, so they only change the level.

The notebook also checks:
1. that one shared map can't replace the nine (each camera left out in turn),
2. why predictions above ~72° are flagged unreliable (Svalbard),
3. how often there is aurora when Model 1 is confident, with and without the 7° shift.

Fits the small model behind the camera-based range in the calibrated notebooks and saves it to
`camera_calibration_maps.json`, which `camera_calibration.py` reads (inputs for the globe and live predictions). To be conservative, no
single camera's map is picked: for any location, Model 1's clear-sky probability goes through all nine maps, and the range
shown is the 25-75% spread of the nine results.

In [1]:
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score

DELTA = 7

bundle = joblib.load("aurora_ebm_model_final_deployment.joblib")
model1, featureslow = bundle["pipeline"], bundle["featureslow"]


def prepare(df):
    df = df.copy()
    df["abs_mlat"] = df["mlat"].abs()
    df["oval_dist"] = df["abs_mlat"] - (66.0 - 2.0 * df["kp"])
    df["cloud_cover"] = 0.0
    return df


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


cams = prepare(pd.read_csv("camera_minutes.csv.gz"))
svalbard = prepare(pd.read_csv("svalbard_minutes.csv.gz"))
order = cams.groupby("camera")["abs_mlat"].mean().sort_values().index

## Model 1, 7° south of each camera

In [2]:
south = cams.copy()
south["mlat"] = np.sign(south["mlat"]) * (south["abs_mlat"] - DELTA)
south["oval_dist"] = south["mlat"].abs() - (66.0 - 2.0 * south["kp"])
cams["p"] = model1.predict_proba(south[featureslow])[:, 1]
cams["p_at_camera"] = model1.predict_proba(cams[featureslow])[:, 1]

cams.groupby("camera").agg(camera_mlat=("abs_mlat", "mean"), observer_mlat=("abs_mlat", lambda m: m.mean() - DELTA),
                           camera_rate=("aurora", "mean"), model1_7_south=("p", "mean"),model1_at_camera_mlat=("p_at_camera","mean")).loc[order].round(3)

,camera_mlat,observer_mlat,camera_rate,model1_7_south,model1_at_camera_mlat
camera,,,,,
chibougamau,58.433,51.433,0.037,0.152,0.503
kapuskasing,58.572,51.572,0.044,0.131,0.454
athabasca,61.187,54.187,0.087,0.269,0.511
the pas,62.406,55.406,0.132,0.312,0.565
gakona,62.930,55.930,0.194,0.296,0.475
whitehorse,63.070,56.070,0.407,0.345,0.479
kiruna,65.034,58.034,0.702,0.451,0.527
skibotn,66.640,59.640,0.841,0.737,0.734
tromso,66.768,59.768,0.554,0.629,0.638


The shift halves the gap between Model 1 predicted probabilities and the cameras (mean gap between Model 1's average and the camera's aurora (positive) rate, 0.27 → 0.12, from `camera_model.ipynb`), but the level still differs from camera to camera: too high at the THEMIS sites, too low at Kiruna. Hence a range across cameras. On average, the levels aren't expected to match exactly. The shift is a calibration, not a better ranking: the AUC within each camera barely moves (median 0.79 → 0.80), but without it Model 1 says >70% far too often at 58-63° (the camera sees aurora in 7-76% of those minutes), with it 68-100% (see "When Model 1 is confident" below).

## One map per camera

In [3]:
def fit_map(df, weights=None):
    return LogisticRegression(C=1e6, max_iter=1000).fit(logit(df["p"].to_numpy())[:, None], df["aurora"],
                                                        sample_weight=weights)


maps = {}
for c in order:
    m = fit_map(cams[cams["camera"] == c])
    maps[c] = {"a": round(float(m.coef_[0, 0]), 3), "b": round(float(m.intercept_[0]), 3),
               "camera_mlat": round(float(cams.loc[cams["camera"] == c, "abs_mlat"].mean()), 1)}
pd.DataFrame(maps).T

,a,b,camera_mlat
chibougamau,1.150,-1.381,58.4
kapuskasing,1.375,-0.662,58.6
athabasca,1.418,-1.297,61.2
the pas,1.815,-1.058,62.4
gakona,1.960,-0.064,62.9
whitehorse,0.992,0.305,63.1
kiruna,1.217,1.444,65.0
skibotn,1.822,0.302,66.6
tromso,1.424,-0.623,66.8


The cameras don't agree, so the calibrated notebooks show a range. How it is built: no map is averaged. Each camera
keeps its own (a, b). A single Model 1 number goes through all nine maps, giving nine "what this camera would record"
numbers, and the range is the middle half of those nine (25-75%). The table answers: **when Model 1 says exactly this
number, what does a typical camera record?**

In [4]:
def apply_map(p, a, b):
    return 1 / (1 + np.exp(-(a * logit(p) + b)))


grid = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
vals = np.array([apply_map(grid, m["a"], m["b"]) for m in maps.values()])
lo, hi = np.quantile(vals, 0.25, axis=0), np.quantile(vals, 0.75, axis=0)
pd.DataFrame({"Model 1 says": [f"{g:.0%}" for g in grid],
              "camera-based range": [f"{l:.0%}-{h:.0%}" for l, h in zip(lo, hi)]}).set_index("Model 1 says")

,camera-based range
Model 1 says,
10%,1%-2%
30%,9%-22%
50%,26%-57%
70%,62%-83%
90%,91%-98%


## Could one shared map do instead?

Leave one camera out: build a shared map from the other eight (cameras weighted equally) and the median of their eight
maps, then score both on the camera left out. Brier score, lower is better.

In [5]:
rows = []
for c in order:
    test, rest = cams[cams["camera"] == c], cams[cams["camera"] != c]
    w = 1.0 / rest.groupby("camera")["aurora"].transform("size")
    shared = fit_map(rest, w * len(rest) / rest["camera"].nunique())
    z = logit(test["p"].to_numpy())[:, None]
    median = np.median([apply_map(test["p"].to_numpy(), m["a"], m["b"]) for k, m in maps.items() if k != c], axis=0)
    rows.append({"camera": c, "raw": brier_score_loss(test["aurora"], test["p"]),
                 "shared map": brier_score_loss(test["aurora"], shared.predict_proba(z)[:, 1]),
                 "median map": brier_score_loss(test["aurora"], median)})
loco = pd.DataFrame(rows).set_index("camera")
print(loco.round(4).to_string())
print("\nmean:", loco.mean().round(4).to_dict())

                raw  shared map  median map
camera                                     
chibougamau  0.0510      0.0395      0.0358
kapuskasing  0.0476      0.0398      0.0388
athabasca    0.1017      0.0880      0.0710
the pas      0.1122      0.0993      0.0813
gakona       0.1201      0.1076      0.1090
whitehorse   0.2100      0.2264      0.2456
kiruna       0.2289      0.2782      0.3136
skibotn      0.1049      0.0962      0.1163
tromso       0.1821      0.1922      0.1730

mean: {'raw': 0.1287, 'shared map': 0.1297, 'median map': 0.1316}


On average neither does better than the raw model (mean Brier 0.1297 and 0.1316 vs 0.1287): they help the southern cameras and hurt others. So instead of one corrected
number, the calibrated notebooks show the 25-75% range across the nine maps (range table above).

## Save

In [6]:
out = {"delta": DELTA, "range_quantiles": [0.25, 0.75], "supported_mlat": [51.0, 60.0], "unreliable_mlat": 72.0,
       "cameras": maps}
with open("camera_calibration_maps.json", "w") as f:
    json.dump(out, f, indent=2)
print(json.dumps(out, indent=2))

{
  "delta": 7,
  "range_quantiles": [
    0.25,
    0.75
  ],
  "supported_mlat": [
    51.0,
    60.0
  ],
  "unreliable_mlat": 72.0,
  "cameras": {
    "chibougamau": {
      "a": 1.15,
      "b": -1.381,
      "camera_mlat": 58.4
    },
    "kapuskasing": {
      "a": 1.375,
      "b": -0.662,
      "camera_mlat": 58.6
    },
    "athabasca": {
      "a": 1.418,
      "b": -1.297,
      "camera_mlat": 61.2
    },
    "the pas": {
      "a": 1.815,
      "b": -1.058,
      "camera_mlat": 62.4
    },
    "gakona": {
      "a": 1.96,
      "b": -0.064,
      "camera_mlat": 62.9
    },
    "whitehorse": {
      "a": 0.992,
      "b": 0.305,
      "camera_mlat": 63.1
    },
    "kiruna": {
      "a": 1.217,
      "b": 1.444,
      "camera_mlat": 65.0
    },
    "skibotn": {
      "a": 1.822,
      "b": 0.302,
      "camera_mlat": 66.6
    },
    "tromso": {
      "a": 1.424,
      "b": -0.623,
      "camera_mlat": 66.8
    }
  }
}


## Svalbard

KHO Svalbard (75.6°; labels from Partamies & Syrjäsuo 2026, *KHOnet2026*, [doi:10.5281/zenodo.19653026](https://doi.org/10.5281/zenodo.19653026), CC BY 4.0) is scored only, with Model 1 at the camera
itself and no 7° shift (Model 1 has already run out of steam with virtually no data around mlat 83°, EBM can't really extrapolate reliably there).

In [7]:
svalbard["p"] = model1.predict_proba(svalbard[featureslow])[:, 1]
print(f"Svalbard: {len(svalbard):,} minutes, aurora fraction {svalbard['aurora'].mean():.2f}, "
      f"Model 1 AUC {roc_auc_score(svalbard['aurora'], svalbard['p']):.2f}")

Svalbard: 145,435 minutes, aurora fraction 0.74, Model 1 AUC 0.55


Close to a coin flip: inside the polar cap. Hence the `unreliable` flag
above ~72°.

## Reassurance check: when Model 1 is confident, is there aurora?

This is a reassurance test. Here we pool every minute above a cutoff (70% means anything from 70% up to 100%).

Take the minutes where Model 1 (for the person 7° south of the camera) says more than
60% or 70%, and ask how often the camera actually saw aurora overhead. That's the precision above a cutoff. Compare it
with the base rate, how often the camera sees aurora at all. The last column shows the same precision without the shift. Svalbard is scored at the camera itself, as above.

In [8]:
def confident(df, cutoff):
    rows = []
    for c, g in df.groupby("camera"):
        sure = g[g["p"] > cutoff]
        rows.append({"camera": c, "mlat": g["abs_mlat"].mean(), "base rate": g["aurora"].mean(),
                     f"% of minutes > {cutoff:.0%}": 100 * len(sure) / len(g), f"minutes > {cutoff:.0%}": len(sure),
                     f"precision > {cutoff:.0%}, Model 1 7° shift": sure["aurora"].mean() if len(sure) else np.nan})
    return pd.DataFrame(rows).set_index("camera")


both = pd.concat([cams, svalbard.assign(p_at_camera=svalbard["p"])])  # Svalbard is scored at the camera, no shift
table = confident(both, 0.6).join(confident(both, 0.7).drop(columns=["mlat", "base rate"]))
table["precision > 70%, Model 1 no shift"] = both[both["p_at_camera"] > 0.7].groupby("camera")["aurora"].mean()
table.loc["kho svalbard", [c for c in table.columns if "7° shift" in c]] = np.nan
table.sort_values("mlat").round(2)

,mlat,base rate,% of minutes > 60%,minutes > 60%,"precision > 60%, Model 1 7° shift",% of minutes > 70%,minutes > 70%,"precision > 70%, Model 1 7° shift","precision > 70%, Model 1 no shift"
camera,,,,,,,,,
chibougamau,58.43,0.04,0.00,0,NaN,0.00,0,NaN,0.07
kapuskasing,58.57,0.04,0.16,152,0.86,0.14,131,1.00,0.14
athabasca,61.19,0.09,5.62,6874,0.57,2.12,2591,0.68,0.30
the pas,62.41,0.13,10.28,9910,0.64,5.89,5674,0.77,0.39
gakona,62.93,0.19,8.27,5919,0.78,4.63,3316,0.94,0.65
whitehorse,63.07,0.41,11.57,6097,0.79,6.30,3320,0.89,0.76
kiruna,65.03,0.70,28.03,23637,0.93,19.73,16638,0.92,0.92
skibotn,66.64,0.84,79.02,18830,0.95,66.07,15745,0.97,0.96
tromso,66.77,0.55,61.51,104324,0.74,43.57,73884,0.81,0.77


In [9]:
auc = cams.groupby("camera").apply(lambda g: pd.Series({
    "AUC no shift": roc_auc_score(g["aurora"], g["p_at_camera"]),
    "AUC 7° shift": roc_auc_score(g["aurora"], g["p"])}), include_groups=False).loc[order]
print(auc.round(3).to_string())
print("\nmedian:", auc.median().round(2).to_dict())

             AUC no shift  AUC 7° shift
camera                                 
chibougamau         0.703         0.715
kapuskasing         0.728         0.732
athabasca           0.786         0.784
the pas             0.869         0.871
gakona              0.820         0.852
whitehorse          0.704         0.709
kiruna              0.793         0.799
skibotn             0.853         0.873
tromso              0.791         0.817

median: {'AUC no shift': 0.79, 'AUC 7° shift': 0.8}


For observers at about 54-60° (cameras at 61-67°), when Model 1 says more than 70% the camera sees aurora 68-97% of the time, above the base rate everywhere and far above it at the THEMIS sites. Lowering the cutoff to 60% roughly doubles the confident minutes at the THEMIS cameras, at a cost of 10-16 points of precision there (Kiruna and Skibotn barely change). Further south Model 1 is almost never this confident, so there is little to judge. At Svalbard precision just matches the base rate: confidence there says nothing. The AUC table shows the other half of the story: the shift barely changes the ranking (median 0.79 → 0.80), it fixes the level.